# Day 32: Hvass-Labs/Finance-Research Day 1/2

**日期**: 2026-08-10 | **阶段**: 补学 Day 6/8 | **时长**: 5小时

## 今日目标
- 运行 Hvass-Labs 核心 Notebooks，理解长期股票预测方法论
- 对比 Hvass-Labs（学术/长期/组合优化）vs Auquan（教程/短期/策略）
- 理解数据驱动投资研究的完整流程

## 仓库结构概览

| 系列 | 主题 | 风格 |
|------|------|------|
| 01-01G | 长期股票预测（7个Notebook逐步深入） | 核心方法论 |
| 02 | 股票指数对比 | 数据分析 |
| 03-04 | 投资组合优化 | 组合管理 |
| 05-07 | 财务比率预测（PSales/营收增长/股利） | 基本面 |
| Paper_* | 已发表SSRN论文 | 学术验证 |
| *.py | 共享模块（data/portfolio/curve_fit） | 工程化 |

## Part 1: 预检 — 加载共享模块

Hvass-Labs 把数据获取和核心算法抽成了 .py 模块，比 Auquan 工程化程度更高。

In [ ]:
import sys
sys.path.insert(0, '/Users/macbookpro/Desktop/股/day32/Finance-Research')

# 尝试导入共享模块
for mod_name in ['data', 'curve_fit', 'portfolio']:
    try:
        exec(f'import {mod_name}')
        print(f'✅ {mod_name}.py 加载成功')
    except Exception as e:
        print(f'❌ {mod_name}.py: {e}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'PingFang SC']
plt.rcParams['axes.unicode_minus'] = False
print('\n✅ 基础环境就绪')

## Part 2: 快速浏览所有 Notebook

不用逐个跑所有 Notebook（太多），做「精读式浏览」，列清单。

In [ ]:
import os, json, glob

repo = '/Users/macbookpro/Desktop/股/day32/Finance-Research'
notebooks = sorted(glob.glob(f'{repo}/*.ipynb'))
print(f'共 {len(notebooks)} 个 Notebook:\n')

for nb_path in notebooks:
    name = os.path.basename(nb_path)
    with open(nb_path) as f:
        nb = json.load(f)
    # 找第一个 markdown cell 的标题
    for cell in nb['cells']:
        if cell['cell_type'] == 'markdown':
            text = ''.join(cell['source'])[:100].replace('\n', ' ')
            print(f'  {name:55s} | {text}')
            break

## Part 3: 深入运行 — S曲线长期股票预测（核心方法论）

这是整个仓库的灵魂：用 **S型曲线 y=a+b/(1+exp(c+d*x))** 拟合股票长期趋势。

**直观理解**：股价长期不是随机游走，而是围绕一条 S 曲线波动。公司不会长到天上。

In [ ]:
from scipy.optimize import curve_fit as scipy_curve_fit
import akshare as ak

# ===== 拉茅台日线 → 重采样为月线 =====
df_daily = ak.stock_zh_a_daily(symbol='sh600519', start_date='20100101', 
                                end_date='20241231', adjust='qfq')
df_daily['date'] = pd.to_datetime(df_daily['date'])
df_daily = df_daily.set_index('date').sort_index()

# 日线 → 月线：取每月最后交易日
df_monthly = df_daily.resample('ME').last().dropna()
df_monthly = df_monthly.reset_index()

dates = df_monthly['date']
x = np.arange(len(df_monthly))
y = df_monthly['close'].values

# S曲线模型：y = a + b / (1 + exp(c + d*x))
def s_curve(x, a, b, c, d):
    return a + b / (1 + np.exp(c + d * x))

# 初始猜测
p0 = [y.min(), y.max() - y.min(), 1, -0.01]

try:
    popt, _ = scipy_curve_fit(s_curve, x, y, p0=p0, maxfev=10000)
    a, b, c, d = popt
    y_fit = s_curve(x, *popt)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.scatter(dates, y, s=15, alpha=0.6, label='茅台月线收盘价')
    ax.plot(dates, y_fit, 'r-', lw=2, label=f'S曲线拟合: y={a:.0f}+{b:.0f}/(1+exp({c:.2f}+{d:.4f}*x))')
    ax.set_title('贵州茅台(600519) 长期走势 S曲线拟合', fontsize=15)
    ax.set_ylabel('收盘价（前复权）'); ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/Users/macbookpro/Desktop/股/day32/maotai_s_curve.png', dpi=100, bbox_inches='tight')
    plt.show()
    print(f'拟合参数: a={a:.1f}, b={b:.1f}, c={c:.3f}, d={d:.6f}')
    print(f'S曲线收敛值: {a+b:.1f} 元（模型预测的长期上限）')
except Exception as e:
    print(f'拟合失败: {e}')

### S曲线核心洞察

| 维度 | Hvass-Labs 方法 | 传统技术分析 |
|------|----------------|-------------|
| 时间尺度 | **月线/年线**（长期） | 日线/周线（短期） |
| 建模对象 | 股价水平（带基本面 S 曲线） | 价格变化（均线/RSI/MACD） |
| 核心假设 | 股价围绕长期趋势波动（均值回归） | 趋势会延续（动量） |
| 数据 | 基本面(PSales/营收)+价格 | 仅价格+成交量 |
| 可证伪性 | 有学术论文+统计检验 | 多为经验法则 |

## Part 4: 中美指数收益率分布对比

Hvass-Labs的对比方法论：对数收益率直方图+正态拟合。

In [ ]:
import akshare as ak

# ===== 拉沪深300日线 =====
hs300 = ak.stock_zh_index_daily(symbol='sh000300')
hs300['date'] = pd.to_datetime(hs300['date'])
hs300 = hs300.set_index('date').sort_index()
hs300_ret = hs300['close'].pct_change().dropna()

# 直方图 + 正态拟合
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(hs300_ret*100, bins=80, density=True, alpha=0.7, color='steelblue', edgecolor='white')

from scipy import stats
mu, sigma = hs300_ret.mean()*100, hs300_ret.std()*100
x_range = np.linspace(mu-4*sigma, mu+4*sigma, 200)
ax.plot(x_range, stats.norm.pdf(x_range, mu, sigma), 'r-', lw=2, 
        label=f'正态拟合 μ={mu:.2f}% σ={sigma:.2f}%')
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_title('沪深300 日收益率分布 vs 正态分布', fontsize=14)
ax.set_xlabel('日收益率(%)'); ax.legend()
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day32/hs300_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'沪深300 日收益: 均值={mu:.3f}%, 波动率={sigma:.2f}%')
print(f'偏度={hs300_ret.skew():.2f}, 峰度={hs300_ret.kurtosis():.2f}')
print(f'正收益天数占比: {(hs300_ret>0).mean():.1%}')
print('\n💡 偏度<0 + 峰度>0 = 左偏肥尾：暴跌比暴涨更常见且更极端')

## Part 5: ⭐ Hvass-Labs vs Auquan 方法论对比（核心产出）

In [ ]:
# ===== 对比总结 =====
comparison = pd.DataFrame({
    '维度': ['投资周期', '核心方法', '数据来源', '模型复杂度', '工程化', 
            '学术严谨', '可复现性', '适合人群', '适合市场', '代码风格'],
    'Hvass-Labs': [
        '长期（月/年级）',
        'S曲线拟合+组合优化',
        'SimFin基本面+价格',
        '简单（S曲线+线性回归）',
        '高（共享.py模块+InvestOps包）',
        '高（SSRN论文+统计检验）',
        '高（数据源公开+方法透明）',
        '长期投资者/研究员',
        '美股（数据依赖SimFin）',
        '模块化，函数封装良好'
    ],
    'Auquan': [
        '短中期（日/周级）',
        '统计检验+ML策略',
        'yfinance价格为主',
        '中（从统计到ML逐步递进）',
        '中（独立Notebook，可复用性弱）',
        '中（教学导向，无同行评审）',
        '中（依赖yfinance可能限流）',
        '量化新手/交易爱好者',
        '通用（教程性质，不绑定市场）',
        '教学式，循序渐进'
    ]
})

print(comparison.to_string(index=False))

print('\n' + '='*60)
print('🎯 关键洞察：两者互补，不是竞争')
print('='*60)
print('• Auquan 教你「怎么量化交易」（工具+策略+陷阱）')
print('• Hvass-Labs 教你「为什么要这样投资」（理论+证据+优化）')
print('• 组合使用：用Hvass方法选股 + 用Auquan方法执行')

## Part 6: 工程化收获

### 从 Hvass-Labs 学到的工程化思路
1. **共享模块抽离**：data.py/curve_fit.py/portfolio.py 让多个 Notebook 复用同一套逻辑
2. **参数集中管理**：data_keys.py 集中管理所有数据字段名
3. **论文级可复现**：Paper_*.ipynb 可以直接生成论文图表

### 投资思路
1. **长期趋势是存在的**：股价围绕S曲线波动，不是完全随机
2. **基本面驱动长期回报**：PSales、营收增长、股利才是长期锚
3. **简单模型+严格验证 > 复杂模型+宽松验证**

## 今日小结
| 项目 | 完成 |
|------|:--:|
| 仓库结构浏览 | ✅ |
| S曲线拟合茅台 | ✅ |
| 沪深300收益率分布 | ✅ |
| Hvass vs Auquan 对比 | ✅ |
| 明天: Day 33 终极练习 | ⬜ |